# TODO 6.3：GPTQ INT4 实验

使用与 AWQ 相同的校准集、bit 数和 group size 32/64/128。完整模型写入 `banking_llm_project/gptq`，分析文件写入 `OptimizationSystem/out/gptq`。

In [1]:
%pip -q install -U gptqmodel optimum accelerate safetensors
# 安装完成后如 Colab 提示重启，请重启运行时再从头运行。

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 62.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 5.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.0/215.0 kB 19.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.6/115.6 kB 10.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
    

In [2]:
from google.colab import drive
drive.mount('/content/drive')
import gc, json, sys
from pathlib import Path
import torch
PROJECT_DIR = Path('/content/drive/MyDrive/OptimizationSystem')
DATASET_DIR = Path('/content/drive/MyDrive/banking_llm_project/datasets')
MODEL_MANIFEST = Path('/content/drive/MyDrive/banking_llm_project/model_input_real/model_manifest.json')
CONFIG_PATH = PROJECT_DIR / 'configs/gptq.json'
CALIBRATION_MANIFEST = PROJECT_DIR / 'out/dataset_input/dataset_manifest.json'
RESULT_DIR = PROJECT_DIR / 'out/gptq'
MODEL_RESULT_DIR = Path('/content/drive/MyDrive/banking_llm_project/gptq')
sys.path.insert(0, str(PROJECT_DIR))
from src.data.finqa_assets import load_assets
from src.quantization.gptq import build_gptq_plans, run_gptq_plan, collect_gptq_evidence, validate_gptq_evidence
assets = load_assets(DATASET_DIR)
config = json.loads(CONFIG_PATH.read_text(encoding='utf-8'))
plans = build_gptq_plans(config)
assert torch.cuda.is_available(), '请切换到 NVIDIA GPU。'
print(torch.cuda.get_device_name(0), [plan.plan_id for plan in plans])

Mounted at /content/drive
NVIDIA A100-SXM4-40GB ['gptq-int4-w4a16-g32', 'gptq-int4-w4a16-g64', 'gptq-int4-w4a16-g128']


In [3]:
results = []
for plan in plans:
    results.append(run_gptq_plan(
        plan=plan, model_manifest_file=MODEL_MANIFEST,
        calibration_prompts=assets.calibration_prompts,
        output_dir=RESULT_DIR / plan.plan_id,
        model_output_dir=MODEL_RESULT_DIR / plan.plan_id,
        max_calibration_samples=int(config['max_calibration_samples']),
        max_calibration_seq_len=int(config['max_calibration_seq_len']),
    ))
    gc.collect(); torch.cuda.empty_cache()
print(json.dumps(results, ensure_ascii=False, indent=2))

[
  {
    "plan": {
      "plan_id": "gptq-int4-w4a16-g32",
      "method": "gptq",
      "bits": 4,
      "weight_bits": 4,
      "activation_bits": 16,
      "group_size": 32,
      "quantization_granularity": "groupwise",
      "backend_package": "gptqmodel"
    },
    "status": "not_run",
    "detail": "ModuleNotFoundError(\"Could not import module 'AutoModelForCausalLM'. Are this object's requirements defined correctly?\")"
  },
  {
    "plan": {
      "plan_id": "gptq-int4-w4a16-g64",
      "method": "gptq",
      "bits": 4,
      "weight_bits": 4,
      "activation_bits": 16,
      "group_size": 64,
      "quantization_granularity": "groupwise",
      "backend_package": "gptqmodel"
    },
    "status": "not_run",
    "detail": "ModuleNotFoundError(\"Could not import module 'AutoModelForCausalLM'. Are this object's requirements defined correctly?\")"
  },
  {
    "plan": {
      "plan_id": "gptq-int4-w4a16-g128",
      "method": "gptq",
      "bits": 4,
      "weight_bits": 4,
  

In [4]:
evidence = collect_gptq_evidence(CONFIG_PATH, CALIBRATION_MANIFEST, MODEL_MANIFEST, RESULT_DIR)
evidence_path = RESULT_DIR / 'gptq_evidence.json'
evidence_path.write_text(json.dumps(evidence, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
report = validate_gptq_evidence(evidence_path)
(RESULT_DIR / 'validation.json').write_text(json.dumps(report.to_dict(), ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
print(json.dumps(report.to_dict(), ensure_ascii=False, indent=2))
assert report.complete, 'GPTQ 三个 group size 未全部完成。'

{
  "evidence": "/content/drive/MyDrive/OptimizationSystem/out/gptq/gptq_evidence.json",
  "complete": false,
  "checks": [
    {
      "name": "gptq_awq_comparable_matrix",
      "status": "pass",
      "detail": "GPTQ uses INT4 group sizes 32, 64, and 128."
    },
    {
      "name": "gptq_backend_recorded",
      "status": "pass",
      "detail": "GPTQModel version is recorded."
    },
    {
      "name": "gptq_quantized_artifacts",
      "status": "not_run",
      "detail": "Each GPTQ plan has config, manifest, and event log."
    },
    {
      "name": "gptq_real_execution",
      "status": "not_run",
      "detail": "All GPTQ plans completed real export."
    }
  ]
}


AssertionError: GPTQ 三个 group size 未全部完成。